# Zarr Stores with Xarray

## Learning Objectives:

- Learn how to read a Zarr store with a group hierarchical structure with `xr.DataTree` and `xarray`'s `"zarr"` engine
- Learn how to select and manipulate underlying `dask` arrays from a Zarr store
- Explore how to use Zarr stores with `xarray` for data analysis and visualization

## What is Zarr?

The Zarr data format is an open, community-maintained format designed for efficient, scalable storage of large N-dimensional arrays. It stores data as compressed and chunked arrays in a format well-suited to parallel processing and cloud-native workflows. 

### Zarr Data Organization:
- **Arrays**: N-dimensional arrays that can be chunked and compressed.
- **Groups**: A container for organizing multiple arrays and other groups with a hierarchical structure.
- **Metadata**: JSON-like metadata describing the arrays and groups, including information about data types, dimensions, chunking, compression, and user-defined key-value fields. 
- **Dimensions and Shape**: Arrays can have any number of dimensions, and their shape is defined by the number of elements in each dimension.
- **Coordinates & Indexing**: Zarr supports coordinate arrays for each dimension, allowing for efficient indexing and slicing.

The diagram below from [the Zarr v3 specification](https://wiki.earthdata.nasa.gov/display/ESO/Zarr+Format) showing the structure of a Zarr store:

![ZarrSpec](https://zarr-specs.readthedocs.io/en/latest/_images/terminology-hierarchy.excalidraw.png)


NetCDF and Zarr share similar terminology and functionality, but the key difference is that NetCDF is a single file, while Zarr is a directory-based “store” composed of many chunked files, making it better suited for distributed and cloud-based workflows.

## Opening a dataset with `xr.open_datatree(engine='zarr')`

Let's open up a precipitation zarr store. This dataset was derived from "GPM_3IMERGHH_07" and "M2T1NXFLX_5.12.4" products and has a group hierarchical structure.
**NOTE** We selected `consolidated=True`, Zarr supports consolidated metadata, which allows you to store all metadata in a single file and can improve performance when reading metadata.

In [ ]:
import xarray as xr

precipitation_store = "https://pub-45a1d62ac8d94c4c89f4dc63681a98ed.r2.dev/precipitation.zarr"

precip_dt = xr.open_datatree(precipitation_store, engine="zarr", chunks={}, consolidated=True)

### Accessing data variables
As with on disk storage methods, data from zarr stores in the cloud can be accessed with either dict-like syntax or method based syntax with the `"zarr"` engine.

In [192]:
precip_dt.observed['precipitation']

<xarray.DataArray 'precipitation' (time: 10, lon: 320, lat: 150)> Size: 2MB
dask.array<open_dataset-precipitation, shape=(10, 320, 150), dtype=float32, chunksize=(5, 160, 75), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 80B 2021-08-29T07:30:00 ... 2021-08-29T16:...
  * lat      (lat) float32 600B 20.05 20.15 20.25 20.35 ... 34.75 34.85 34.95
  * lon      (lon) float32 1kB -109.9 -109.8 -109.8 ... -78.25 -78.15 -78.05
Attributes:
    LongName:          \nComplete merged microwave-infrared (gauge-adjusted)\...
    Units:             mm/hr
    units:             mm/hr
    CodeMissingValue:  -9999.9
    DimensionNames:    time,lon,lat

This returns an `xr.DataArray` object with the underly data as a chunked `dask.Array`.

## Time slicing

Like a dataset store on disk we can do time slicing on our zarr store. Each time slice is one hour data with a total of 10 hours of data. 

Let's try getting the first 5 hours of data with `.sel(time=)`. 

Since the time slices are ordered we can get a subset of the array of our time coordinate and pass it to the `.sel` method.

In [ ]:
time_index = precip_dt.time[0:5]
precip_dt.sel(time=time_index)


<xarray.DataTree>
Group: /
│   Dimensions:  (time: 5)
│   Coordinates:
│     * time     (time) datetime64[ns] 40B 2021-08-29T07:30:00 ... 2021-08-29T11:...
├── Group: /observed
│       Dimensions:        (time: 5, lon: 320, lat: 150)
│       Coordinates:
│         * lat            (lat) float32 600B 20.05 20.15 20.25 ... 34.75 34.85 34.95
│         * lon            (lon) float32 1kB -109.9 -109.8 -109.8 ... -78.15 -78.05
│       Data variables:
│           precipitation  (time, lon, lat) float32 960kB dask.array<chunksize=(5, 160, 75), meta=np.ndarray>
└── Group: /reanalysis
        Dimensions:        (time: 5, lat: 31, lon: 52)
        Coordinates:
          * lat            (lat) float64 248B 20.0 20.5 21.0 21.5 ... 34.0 34.5 35.0
          * lon            (lon) float64 416B -110.0 -109.4 -108.8 ... -78.75 -78.12
        Data variables:
            precipitation  (time, lat, lon) float32 32kB dask.array<chunksize=(5, 31, 52), meta=np.ndarray>

We can also subset by time with a `datetime` string.

In [253]:
precip_dt.sel(time=slice('2021-08-29T07:30:00', '2021-08-29T16:30:00'))

<xarray.DataTree>
Group: /
│   Dimensions:  (time: 10)
│   Coordinates:
│     * time     (time) datetime64[ns] 80B 2021-08-29T07:30:00 ... 2021-08-29T16:...
├── Group: /observed
│       Dimensions:        (time: 10, lon: 320, lat: 150)
│       Coordinates:
│         * lat            (lat) float32 600B 20.05 20.15 20.25 ... 34.75 34.85 34.95
│         * lon            (lon) float32 1kB -109.9 -109.8 -109.8 ... -78.15 -78.05
│       Data variables:
│           precipitation  (time, lon, lat) float32 2MB dask.array<chunksize=(5, 160, 75), meta=np.ndarray>
└── Group: /reanalysis
        Dimensions:        (time: 10, lat: 31, lon: 52)
        Coordinates:
          * lat            (lat) float64 248B 20.0 20.5 21.0 21.5 ... 34.0 34.5 35.0
          * lon            (lon) float64 416B -110.0 -109.4 -108.8 ... -78.75 -78.12
        Data variables:
            precipitation  (time, lat, lon) float32 64kB dask.array<chunksize=(10, 31, 52), meta=np.ndarray>

Or by the index of our time dimension `.isel(time=slice())`

In [237]:
precip_dt.isel(time=slice(0, 5))

<xarray.DataTree>
Group: /
│   Dimensions:  (time: 5)
│   Coordinates:
│     * time     (time) datetime64[ns] 40B 2021-08-29T07:30:00 ... 2021-08-29T11:...
├── Group: /observed
│       Dimensions:        (time: 5, lon: 320, lat: 150)
│       Coordinates:
│         * lat            (lat) float32 600B 20.05 20.15 20.25 ... 34.75 34.85 34.95
│         * lon            (lon) float32 1kB -109.9 -109.8 -109.8 ... -78.15 -78.05
│       Data variables:
│           precipitation  (time, lon, lat) float32 960kB dask.array<chunksize=(5, 160, 75), meta=np.ndarray>
└── Group: /reanalysis
        Dimensions:        (time: 5, lat: 31, lon: 52)
        Coordinates:
          * lat            (lat) float64 248B 20.0 20.5 21.0 21.5 ... 34.0 34.5 35.0
          * lon            (lon) float64 416B -110.0 -109.4 -108.8 ... -78.75 -78.12
        Data variables:
            precipitation  (time, lat, lon) float32 32kB dask.array<chunksize=(5, 31, 52), meta=np.ndarray>

### Chunking
Chunking is the process of dividing arrays into smaller pieces, which allows for parallel processing and efficient storage.

To examine the chunks in our Zarr store, with `xarray` you can use the `chunks` attribute on the `xr.DataArray` object.

In [204]:
precip_dt.observed['precipitation'].data.chunks

((5, 5), (160, 160), (75, 75))

#### Selecting by chunks

Since our underlying data arrays are `dask.Array`, we can access data from each chunked array in our Zarr store. 

Let's get the first chunk of the "observed/precipitation" variable in our zarr store.

We add `.data` to our `xr.DataArray` to get access the `dask.Array`. The `.blocks[]` method allows you to index by chunk and `.compute()` returns the `np.ndarray`. 

In [ ]:
precip_dt.observed['precipitation'].data.blocks[0,0,0].compute()

array([[[0.   , 0.   , 0.   , ..., 0.   , 0.   , 0.   ],
        [0.   , 0.   , 0.   , ..., 0.   , 0.   , 0.   ],
        [0.   , 0.   , 0.   , ..., 0.   , 0.   , 0.   ],
        ...,
        [0.   , 0.   , 0.   , ..., 0.   , 0.   , 0.   ],
        [0.   , 0.   , 0.   , ..., 0.   , 0.   , 0.   ],
        [0.   , 0.   , 0.   , ..., 0.   , 0.   , 0.   ]],

       [[0.   , 0.   , 0.   , ..., 0.   , 0.   , 0.   ],
        [0.   , 0.   , 0.   , ..., 0.   , 0.   , 0.   ],
        [0.   , 0.   , 0.   , ..., 0.   , 0.   , 0.   ],
        ...,
        [0.   , 0.   , 0.   , ..., 0.   , 0.   , 0.   ],
        [0.   , 0.   , 0.   , ..., 0.   , 0.   , 0.   ],
        [0.   , 0.   , 0.   , ..., 0.   , 0.   , 0.   ]],

       [[0.   , 0.   , 0.   , ..., 0.   , 0.   , 0.   ],
        [0.   , 0.   , 0.   , ..., 0.   , 0.   , 0.   ],
        [0.   , 0.   , 0.   , ..., 0.   , 0.   , 0.   ],
        ...,
        [0.   , 0.   , 0.   , ..., 0.035, 0.   , 0.   ],
        [0.   , 0.   , 0.   , ..., 0.02 , 0.0

In [251]:
precip_dt.observed['precipitation'].mean(dim='time')

<xarray.DataArray 'precipitation' (lon: 320, lat: 150)> Size: 192kB
dask.array<mean_agg-aggregate, shape=(320, 150), dtype=float32, chunksize=(160, 75), chunktype=numpy.ndarray>
Coordinates:
  * lat      (lat) float32 600B 20.05 20.15 20.25 20.35 ... 34.75 34.85 34.95
  * lon      (lon) float32 1kB -109.9 -109.8 -109.8 ... -78.25 -78.15 -78.05

::::{admonition} Exercise
:class: tip

Can you calculate and plot the mean precipitation starting at 09:55 for the reanalysis group in this zarr store?

:::{admonition} Solution
:class: dropdown

```python
precip_dt.reanalysis['precipitation'].sel(time=slice('2021-08-29T09:55:00', '2021-08-29T16:30:00')).mean(dim='time').plot()
```
:::
::::